# TwoTank - Stage 5b - Safety Evaluation

Evaluate the configured controllers on matched scenarios.


In [12]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

_p = Path.cwd()
while not (_p / 'src' / 'sdpc').exists():
    _p = _p.parent
sys.path.insert(0, str(_p / 'src'))

import matplotlib.pyplot as plt
import pandas as pd
import torch

from sdpc.config import load_config
from sdpc.eval import evaluate_twotank_safety
from sdpc.plotting import plot_states_and_controls
from sdpc.registry import make_system
from sdpc.sindy import load_model

device = torch.device('cpu')
system = make_system('twotank', device=device)
CONFIGS = Path.cwd().parent / 'configs'
RESULTS = Path.cwd().parent / 'results'
cfg = load_config(CONFIGS / 'eval_safety.yaml')
print({k: cfg[k] for k in ('n_trajectories', 'nsteps', 'n_references', 'perturbation')})

{'n_trajectories': 10, 'nsteps': 1000, 'n_references': 4, 'perturbation': {'c1': 0.12, 'c2': 0.02}}


In [ ]:
from sdpc.io import find_policy_checkpoint
policy_path = find_policy_checkpoint(RESULTS, cfg)
sparse_policy = load_model(policy_path, device=device).eval()
print('Sparse policy:', policy_path)

## Run the configured perturbed safety trajectories

Create the deployment model.


In [14]:
safety_out = evaluate_twotank_safety(
    system, sparse_policy, cfg, device, progress=True
)
scenario = safety_out['scenarios']
assert scenario['xn'].shape == (cfg['n_trajectories'], 1, system.nx)
assert scenario['r'].shape == (cfg['n_trajectories'], cfg['nsteps'] + 1, system.nx)
assert torch.all(scenario['r'][..., 0] == scenario['r'][..., 1])
levels = scenario['reference_levels']


Safety trajectory 1/10
Safety trajectory 2/10
Safety trajectory 3/10
Safety trajectory 4/10
Safety trajectory 5/10
Safety trajectory 6/10
Safety trajectory 7/10
Safety trajectory 8/10
Safety trajectory 9/10
Safety trajectory 10/10


In [ ]:
safe_lo, safe_hi = map(float, scenario['state_bounds'].detach().cpu().tolist())
ref_cfg = cfg['reference']
if 'edge_fractions' in ref_cfg:
    edge_values = [safe_lo + (safe_hi - safe_lo) * float(f) for f in ref_cfg['edge_fractions']]
else:
    edge_values = ref_cfg['edge_levels']
for edge in edge_values:
    edge_value = torch.as_tensor(edge, dtype=levels.dtype, device=levels.device)
    assert torch.all(torch.isclose(levels, edge_value).any(dim=1))
print('Distinct initial states:', torch.unique(scenario['xn'][:, 0], dim=0).shape[0])

## Aggregate tracking, smoothness, safety, and computation metrics

Compare the configured methods.


In [ ]:
summary = pd.DataFrame.from_dict(safety_out['summary'], orient='index')
summary.index.name = 'method'
summary_columns = [
    'tracking_mse_mean', 'tracking_mse_std',
    'control_smoothness_rms_mean', 'max_du_l2_mean',
    'num_state_violations_mean', 'num_control_rate_violations_mean',
    'min_state_margin_mean', 'min_control_rate_margin_mean',
    'online_per_step_ms_mean', 'policy_forward_per_step_ms_mean',
    'total_adaptation_iters_mean', 'mean_filter_iters_mean',
]
display(summary.reindex(columns=summary_columns))

## Violations summarized across trajectories

Compare the configured methods.


In [ ]:
safety_violations = pd.DataFrame.from_dict(safety_out['violations'], orient='index')
safety_violations.index.name = 'method'
display(safety_violations)

## Per-trajectory violation distribution

Visualize the closed-loop response.


In [ ]:
safety_rows = pd.DataFrame([
    row for rows in safety_out['per_trajectory'].values() for row in rows
])
display(safety_rows.head())

methods = list(safety_out['per_trajectory'])
counts = [
    [row['num_violations'] for row in safety_out['per_trajectory'][method]]
    for method in methods
]
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot(counts, tick_labels=methods, showmeans=True)
ax.set_ylabel('violating samples per trajectory')
ax.grid(True, axis='y', linestyle='--', alpha=0.4)
fig.tight_layout()
plt.show()

## First matched safety scenario

Compare the executed trajectories.


In [ ]:
safety_out["per_trajectory"]

In [ ]:
examples = safety_out['examples']
styles = {
    'frozen': ('gray', ':'), 'unconstrained': ('darkorange', '--'),
    'barrier': ('green', '-'), 'psf': ('crimson', '-.'),
}
plot_states_and_controls(
    [
        {'x': examples[name]['x_traj'], 'u': examples[name]['u_traj'],
         'label': name, 'color': styles[name][0], 'linestyle': styles[name][1]}
        for name in styles
    ],
    r_traj=scenario['r'][:1], xmin=safe_lo, xmax=safe_hi,
    state_margin=float(cfg.get('bands', {}).get('box', 0.0)),
    title='TwoTank safety evaluation: matched trajectory 0',
)
plt.show()

## All barrier trajectories and violations

Run predictive safety adaptation.


In [ ]:
barrier_traj = safety_out['trajectories']['barrier']
X = barrier_traj['x_traj']
U = barrier_traj['u_traj']
safe_lo, safe_hi = map(float, scenario['state_bounds'].detach().cpu().tolist())
box_band = float(cfg.get('bands', {}).get('box', 0.0))
du_band = float(cfg.get('bands', {}).get('du', 0.0))
du_max = cfg.get('du_max')
tol = 1.0e-7

barrier_failures = safety_rows[
    (safety_rows['method'] == 'barrier') & (safety_rows['num_violations'] > 0)
][['trajectory', 'num_state_violations', 'num_control_rate_violations',
   'min_state_margin', 'min_control_rate_margin']]
display(barrier_failures if len(barrier_failures) else 'No hard barrier violations')

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
colors = plt.colormaps['viridis'](torch.linspace(0.05, 0.95, X.shape[0]).numpy())
for traj_idx, color in enumerate(colors):
    for state_idx in range(system.nx):
        values = X[traj_idx, :, state_idx]
        axes[state_idx].plot(values, color=color, alpha=0.65, linewidth=0.9)
        bad = (values < safe_lo - tol) | (values > safe_hi + tol)
        if bad.any():
            steps = torch.where(bad)[0]
            axes[state_idx].scatter(steps, values[steps], color='red', s=12, zorder=4)

du_l2 = torch.linalg.vector_norm(U[:, 1:] - U[:, :-1], dim=-1)
for traj_idx, color in enumerate(colors):
    axes[2].plot(torch.arange(1, U.shape[1]), du_l2[traj_idx],
                 color=color, alpha=0.65, linewidth=0.9)
    if du_max is not None:
        bad = du_l2[traj_idx] > float(du_max) + tol
        if bad.any():
            steps = torch.where(bad)[0] + 1
            axes[2].scatter(steps, du_l2[traj_idx, bad], color='red', s=12, zorder=4)

for state_idx in range(system.nx):
    axes[state_idx].axhline(safe_lo, color='red', linestyle='--', label='hard bound')
    axes[state_idx].axhline(safe_hi, color='red', linestyle='--')
    if box_band > 0.0:
        axes[state_idx].axhline(safe_lo + box_band, color='darkorange', linestyle=':', label='barrier band')
        axes[state_idx].axhline(safe_hi - box_band, color='darkorange', linestyle=':')
    axes[state_idx].set_ylabel(f'x{state_idx}')
    axes[state_idx].grid(True, linestyle='--', alpha=0.35)
    axes[state_idx].legend(loc='best')
if du_max is not None:
    axes[2].axhline(float(du_max), color='red', linestyle='--', label='du_max')
    if 0.0 < du_band < float(du_max) ** 2:
        du_band_threshold = (float(du_max) ** 2 - du_band) ** 0.5
        axes[2].axhline(du_band_threshold, color='darkorange', linestyle=':', label='barrier band')
axes[2].set_ylabel(r'$\|u_k-u_{k-1}\|_2$')
axes[2].set_xlabel('time step')
axes[2].grid(True, linestyle='--', alpha=0.35)
axes[2].legend(loc='best')
fig.suptitle('Barrier adaptation: all trajectories (red markers are hard violations)')
fig.tight_layout()
plt.show()

## Save tables and all trajectories

Save the generated artifacts.


In [ ]:
out_dir = RESULTS / 'eval' / f"safety_{cfg['n_trajectories']}x{cfg['nsteps']}"
out_dir.mkdir(parents=True, exist_ok=True)
summary.to_csv(out_dir / 'summary.csv')
safety_violations.to_csv(out_dir / 'violation_summary.csv')
safety_rows.to_csv(out_dir / 'per_trajectory.csv', index=False)
trajectory_payload = {
    'trajectories': safety_out['trajectories'],
    'scenarios': {
        key: value.detach().cpu() if isinstance(value, torch.Tensor) else value
        for key, value in safety_out['scenarios'].items()
    },
    'safety': {
        'xmin': safe_lo,
        'xmax': safe_hi,
        'du_max': cfg.get('du_max'),
        'bands': cfg.get('bands', {}),
    },
}
torch.save(trajectory_payload, out_dir / 'all_trajectories.pt')
print('Saved tables and all trajectories to', out_dir)